# Notebook para extraer la matriz de generación

En la plataforma del AMM encontramos una "api" para extraer esta información mediante una sesión TEMPORAL dada una cookie de usuario.

Primero localizamos la COOKIE:
- Pasos:
    1. Entramos a la siguiente página: https://reportesbi.amm.org.gt/knowage/servlet/AdapterHTTP?PAGE=LoginPage&NEW_SESSION=TRUE#
    2. Presionamos la tecla **F12**.
    3. Se desplegará una barra de desarrollador, donde haremos click en la pestaña "network".

    <img src="img/instrP1.png" alt="alt text" width="700">

    4. Con esta pestaña abierta, daremos click en el menú desplegable del AMM en la pestaña "Generación" y luego daremos click en la segunda pestaña llamada "Generación por tecnología".

    <img src="img/instr1.png" alt="alt text" width="700">

    5. Al cargar la página, filtramos la gráfica de tiempos de ejecución al bloque de mayor carga.
    6. Debajo de la gráfica, en la columna "Name" apareceran los procesos cargados por la página, aqui buscaremos uno que empiece por "data?offset" y le daremos click.

    <img src="img/instrP2.png" alt="alt text" width="700">

    7. Comprobamos en la pestaña "Response", que se despliega al dar click en el proceso, que la consulta es un archivo JSON con los datos que vamos a extraer. 

    <img src="img/instrP3.png" alt="alt text" width="700">

    8. Una vez encontramos el proceso con el "Response" correcto, nos dirigimos a su pestaña de "Headers" donde observaremos el url de la sesión y lo copiaremos.

    <img src="img/instrP4.png" alt="alt text" width="700">

    9. Bajando un poco más encontraremos nuestra cookie en el apartado "Cookie" y copiaremos únicamente la primera de nombre "JSESSIONID" hasta antes del ";".
    10. Un poco más abajo, en el apartado "Referer" encontraremos el url de la api, donde copiaremos hasta donde dice "/execute".

    <img src="img/instrP5.png" alt="alt text" width="700">
    
    11. Para verificar la consulta que hace la página, podemos verificar en la pestaña "Payload" del proceso en el apartado "Request Payload".
    12. El paso 8 y 10 se realiza una sola vez, a menos que el scripts este fallando por cambios en el AMM.
    13. Del paso 1 al 7 y el paso 9 se realiza cada que se requiera extraer datos.
    14. Cabe resaltar que este método es TEMPORAL y solo funciona si la sesión dada la cookie no expira.     


## Código

Librerias necesarias, versión Python 3.14.3

In [1]:
import pandas as pd
import requests
import datetime as dt
import urllib3
import calendar

In [5]:
def formatear_df(df_crudo: pd.DataFrame, fecha_reporte: str) -> pd.DataFrame:
    '''
        Función para formatear el DataFrame crudo obtenido de Knowage, limpiando y transformando los datos
        para obtener un DataFrame con las columnas específicas requeridas para el csv final.

        Columnas finales requeridas:
        - fecha
        - turbina_vapor
        - turbina_gas
        - biomasa
        - geotermica
        - solar
        - eolica
        - plantas_hidraulicas
        - motores_reciprocantes
        - importaciones_sin
        - exportaciones_sin
        - demanda_maxima_sni
        - subtotal_generacion_sni
    '''
    # Convertir 'column_3' a valores numéricos (float)
    # errors='coerce' transforma strings no válidos en NaN, y fillna(0) los convierte en 0
    df_crudo['column_3'] = pd.to_numeric(df_crudo['column_3'], errors='coerce').fillna(0)

    # 1. Diccionario para mapear los nombres del origen al formato de las columnas finales
    mapeo_tecnologias = {
        'turbina de vapor': 'turbina_vapor',
        'turbina de gas': 'turbina_gas',
        'biogas': 'biomasa', 
        'geotérmica': 'geotermica',
        'fotovoltaica': 'solar',
        'eólico': 'eolica',
        'hidroeléctrica': 'plantas_hidraulicas',
        'motor reciprocante': 'motores_reciprocantes',
        'importacion': 'importaciones_sin',
        'exportacion': 'exportaciones_sin'
    }

    # Limpiar la 'column_1': quitar el caracter ¨, espacios en blanco y convertir a minúsculas
    df_crudo['tech_limpia'] = df_crudo['column_1'].astype(str).str.replace('¨', '', regex=False).str.strip().str.lower()
    df_crudo['tech_mapeada'] = df_crudo['tech_limpia'].map(mapeo_tecnologias).fillna(df_crudo['tech_limpia'])

    # 2. Calcular Demanda Máxima SNI
    # Agrupar por la hora (column_2 en str) y sumar la potencia ya convertida a float
    demanda_maxima_sni = df_crudo.groupby('column_2')['column_3'].sum().max()

    # 3. Sumar la potencia diaria agrupando por la tecnología ya mapeada
    suma_diaria = df_crudo.groupby('tech_mapeada')['column_3'].sum().to_dict()

    # 4. Definir las columnas deseadas manteniendo el orden solicitado
    tecnologias_generacion = [
        'turbina_vapor','turbina_gas','biomasa', 'geotermica', 'solar', 
        'eolica', 'plantas_hidraulicas', 'motores_reciprocantes'
    ]
    interconexiones = ['importaciones_sin', 'exportaciones_sin']
    
    todas_las_columnas = tecnologias_generacion + interconexiones

    # 5. Construir la fila resultante inicializando en 0 las tecnologías ausentes
    fila_resultado = {'fecha': fecha_reporte}
    
    for col in todas_las_columnas:
        fila_resultado[col] = suma_diaria.get(col, 0.0)

    # 6. Calcular Subtotal Generación SNI (excluyendo importaciones y exportaciones)
    fila_resultado['subtotal_generacion_sni'] = sum(fila_resultado[tech] for tech in tecnologias_generacion)
    fila_resultado['demanda_maxima_sni'] = demanda_maxima_sni

    # 7. Convertir el diccionario a un DataFrame con el orden específico requerido
    orden_columnas = ['fecha'] + todas_las_columnas + ['demanda_maxima_sni', 'subtotal_generacion_sni']
    
    df_final = pd.DataFrame([fila_resultado], columns=orden_columnas)
    
    return df_final

def extract(anio:int,mes:int,dia:int,session:requests.Session,url:str,payload:dict) -> pd.DataFrame:
    '''Función para extraer los datos de generación eléctrica del SNI para una fecha específica.
    
    ### Parámetros:
    - **anio** (int): Año de la fecha a extraer.
    - **mes** (int): Mes de la fecha a extraer.
    - **dia** (int): Día de la fecha a extraer.
    - **session** (requests.Session): Sesión de requests para realizar la petición HTTP.
    - **url** (str): URL del endpoint de Knowage para la extracción de datos.
    - **payload** (dict): Payload base para la petición, que se actualizará con los parámetros de fecha.
    '''
    payload["parameters"]["anio"]=anio # seteamos año
    payload["parameters"]["mes"]=mes # seteamos mes
    payload["parameters"]["dia"]=dia # seteamos día 
    try:
        response = session.post(url, json=payload, timeout=25, verify=False) # Enviamos el payload en formato JSON
        
        if response.status_code == 200: # códgio 200 signifíca OK, response enviado
            datos_json = response.json() # capturamos el response 
            
            # En Knowage, 'rows' suele ser la lista de registros
            filas = datos_json.get('rows', []) # obtenemos los datos en lista dado el parámetro "rows" del JSON del response
            
            if isinstance(filas, list) and len(filas) > 0: # si no esta vacía entonces procedemos
                df = pd.DataFrame(filas) # convertimos a dataframe los datos

                print(f"Extrayendo año {anio}, mes {mes} y día {dia}")
                df = formatear_df(df,dt.date(anio,mes,dia).strftime("%d/%m/%Y"))
                print("Extracción exitosa.")
                return df
            else:
                print("La llave 'rows' está vacía o no es una lista.")
                # Ver una muestra de 'rows' para entender su formato
                print("Contenido de 'rows':", str(filas)[:200]) 
                
        else:
            print(f"Error del servidor: {response.status_code}")

    except Exception as e:
        print(f"Error al procesar el DataFrame: {e}")

In [3]:
def process_extract(anio:int,mes:int,dia:int,cookie:str) -> pd.DataFrame:
    '''
    Función principal para gestionar el proceso de extracción de datos para una fecha específica.
    ### Parámetros:
    - **anio** (int): Año de la fecha a extraer.
    - **mes** (int): Mes de la fecha a extraer.
    - **dia** (int): Día de la fecha a extraer.
    - **cookie** (str): Valor de la cookie JSESSIONID para autenticación en la sesión de Knowage.
    '''
    # Desactivar advertencias de SSL si usas verify=False
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    # Url de la sesión/proceso, no cambiar
    url = "https://reportesbi.amm.org.gt/knowage/restful-services/2.0/datasets/Generacion%20-%20Generacion%20por%20Tecnologia/data?offset=-1&size=-1&nearRealtime=true&widgetName=new%20Widget"

    # Iniciar una sesión persistente
    session = requests.Session()

    # link de la "api"
    session.headers.update({
        'Referer': 'https://reportesbi.amm.org.gt/knowagecockpitengine/api/1.0/pages/execute'
    })

    # Seteamos Cookie
    session.headers.update({'Cookie': f"JSESSIONID={cookie}"}) # cambiamos después del "="

    # Payload del proceso (lo que vamos a solicitar y de dónde)
    payload = {
        "aggregations": {
            "measures": [{"id": "POTENCIA", "alias": "Potencia", "columnName": "POTENCIA", "funct": "SUM", "orderType": ""}],
            "categories": [
                {"id": "TIPO_GENERACION", "alias": "Tipo Generacion", "columnName": "TIPO_GENERACION", "orderType": ""},
                {"id": "HI", "alias": "Hora Inicial", "columnName": "HI", "orderType": ""}
            ],
            "dataset": "Generacion - Generacion por Tecnologia"
        },
        "parameters": {"anio": "", "mes": "", "dia": ""}, # parámetros vacios para modificar luego
        "selections": {}
    }

    df = extract(anio,mes,dia,session,url,payload)
    return df

In [6]:
process_extract(2024,1,1,"BE3D9E522F972738936BDAC5956D2DA2")

Extrayendo año 2024, mes 1 y día 1
Extracción exitosa.


,fecha,turbina_vapor,turbina_gas,biomasa,geotermica,solar,eolica,plantas_hidraulicas,motores_reciprocantes,importaciones_sin,exportaciones_sin,demanda_maxima_sni,subtotal_generacion_sni
0,01/01/2024,12821.4325,0.0,0.0,784.185,622.65,1788.1875,9769.4575,24.54,793.485,-1441.925,1400.18,25810.4525


### Extracción masiva

In [ ]:
import requests
import urllib3
import pandas as pd
import datetime as dt
import time
import random
import calendar
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# 1. Configuración inicial y desactivación de advertencias
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://reportesbi.amm.org.gt/knowage/restful-services/2.0/datasets/Generacion%20-%20Generacion%20por%20Tecnologia/data?offset=-1&size=-1&nearRealtime=true&widgetName=new%20Widget"

# 2. Configuración de Sesión y Pool de Conexiones
MAX_WORKERS = 10  

# Instanciar la sesión global
session = requests.Session()
session.headers.update({
    'Referer': 'https://reportesbi.amm.org.gt/knowagecockpitengine/api/1.0/pages/execute',
    'Cookie': "JSESSIONID=36869560D0FEB5F6B83C9433804A4DA8" # Actualice este valor si caduca
})

# Ampliar el pool de conexiones para multithreading
retry_strategy = Retry(
    total=3,
    backoff_factor=0.5, 
    status_forcelist=[429, 500, 502, 503, 504]
)
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS, max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

# 3. Función a ejecutar por cada hilo
def extraer_datos_dia(fecha):
    max_intentos = 3
    intento_actual = 0
    
    payload = {
        "aggregations": {
            "measures": [{"id": "POTENCIA", "alias": "Potencia", "columnName": "POTENCIA", "funct": "SUM", "orderType": ""}],
            "categories": [
                {"id": "TIPO_GENERACION", "alias": "Tipo Generacion", "columnName": "TIPO_GENERACION", "orderType": ""},
                {"id": "HI", "alias": "Hora Inicial", "columnName": "HI", "orderType": ""}
            ],
            "dataset": "Generacion - Generacion por Tecnologia"
        },
        "parameters": {
            "anio": str(fecha.year),
            "mes": str(fecha.month),
            "dia": str(fecha.day)
        },
        "selections": {}
    }

    while intento_actual < max_intentos:
        # Desfase para no saturar al servidor de golpe
        time.sleep(random.uniform(0.5, 1.5) + (intento_actual * 2)) 
        
        try:
            # CORRECCIÓN: Usar session.post en lugar de requests.post. 
            # No se pasa 'headers' porque ya están en session.headers.
            response = session.post(url, json=payload, timeout=30, verify=False)
            
            if response.status_code == 200:
                datos_json = response.json()
                filas = datos_json.get('rows', [])
                
                if isinstance(filas, list) and len(filas) > 0:
                    df = pd.DataFrame(filas)
                    # Asegúrese de que formatear_df esté definida antes de ejecutar esto
                    df = formatear_df(df, fecha.strftime("%d/%m/%Y"))
                    return df
                return None
                
            elif response.status_code == 400:
                print(f"Error HTTP 400 fatal para {fecha.date()}. Revise JSESSIONID o formato.")
                return None 
                
            else:
                intento_actual += 1
                print(f"Intento {intento_actual} fallido para {fecha.date()}: HTTP {response.status_code}")
                
        except (requests.exceptions.ConnectionError, requests.exceptions.ChunkedEncodingError) as e:
            intento_actual += 1
            print(f"Corte de conexión en {fecha.date()}. Reintentando ({intento_actual}/{max_intentos})...")
            
        except Exception as e:
            print(f"Error crítico en {fecha.date()}: {type(e).__name__}")
            return None

    print(f"Descartando {fecha.date()} tras {max_intentos} intentos.")
    return None

# 4. Generación eficiente de fechas y Ejecución Concurrente
if __name__ == "__main__":
    fechas = pd.date_range(start="2021-01-01", end="2026-12-31")
    dfs = []

    print(f"Iniciando extracción concurrente con {MAX_WORKERS} hilos...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros = {executor.submit(extraer_datos_dia, fecha): fecha for fecha in fechas}
        
        for futuro in as_completed(futuros):
            fecha_procesada = futuros[futuro]
            try:
                df_resultado = futuro.result()
                if df_resultado is not None and not df_resultado.empty:
                    dfs.append(df_resultado)
                    print(f"Éxito: {fecha_procesada.date()}")
            except Exception as e:
                print(f"Excepción no controlada en {fecha_procesada.date()}: {e}")

    # 5. Consolidación de datos
    if dfs:
        final = pd.concat(dfs, ignore_index=True)
        final.sort_values(by='fecha', inplace=True)  # Ordenar por fecha
        final.to_csv("generacion.csv", index=False)
        print(f"Extracción finalizada. Registros totales: {len(final)}")
    else:
        print("No se extrajeron datos.")

Iniciando extracción concurrente con 10 hilos...
Éxito: 2021-01-05
Éxito: 2021-01-08
Éxito: 2021-01-10
Éxito: 2021-01-06
Éxito: 2021-01-07
Éxito: 2021-01-02
Éxito: 2021-01-03
Éxito: 2021-01-04
Éxito: 2021-01-01
Éxito: 2021-01-09
Éxito: 2021-01-14
Éxito: 2021-01-15
Éxito: 2021-01-12
Éxito: 2021-01-11
Éxito: 2021-01-18
Éxito: 2021-01-16
Éxito: 2021-01-17
Éxito: 2021-01-19
Éxito: 2021-01-13
Éxito: 2021-01-21
Éxito: 2021-01-20
Éxito: 2021-01-27
Éxito: 2021-01-25
Éxito: 2021-01-22
Éxito: 2021-01-24
Éxito: 2021-01-23
Éxito: 2021-01-26
Éxito: 2021-01-29
Éxito: 2021-01-30
Éxito: 2021-01-28
Éxito: 2021-01-31
Éxito: 2021-02-05
Éxito: 2021-02-01
Éxito: 2021-02-02
Éxito: 2021-02-06
Éxito: 2021-02-04
Éxito: 2021-02-03
Éxito: 2021-02-08
Éxito: 2021-02-09
Éxito: 2021-02-12
Éxito: 2021-02-07
Éxito: 2021-02-13
Éxito: 2021-02-14
Éxito: 2021-02-16
Éxito: 2021-02-10
Éxito: 2021-02-11
Éxito: 2021-02-19
Éxito: 2021-02-20
Éxito: 2021-02-21
Éxito: 2021-02-18
Éxito: 2021-02-15
Éxito: 2021-02-17
Éxito: 2021-02-

In [14]:
gen = pd.read_csv("generacion.csv")

In [17]:
gen["fecha"] = pd.to_datetime(gen["fecha"], format="%d/%m/%Y")

In [18]:
gen.set_index("fecha",drop=True,inplace=True)

In [23]:
gen.sort_index(inplace=True)

In [29]:
gen.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1967 entries, 2021-01-01 to 2026-05-21
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   turbina_vapor            1967 non-null   float64
 1   turbina_gas              1967 non-null   float64
 2   geotermica               1967 non-null   float64
 3   solar                    1967 non-null   float64
 4   eolica                   1967 non-null   float64
 5   plantas_hidraulicas      1967 non-null   float64
 6   motores_reciprocantes    1967 non-null   float64
 7   importaciones_sin        1967 non-null   float64
 8   exportaciones_sin        1967 non-null   float64
 9   demanda_maxima_sni       1967 non-null   float64
 10  subtotal_generacion_sni  1967 non-null   float64
dtypes: float64(11)
memory usage: 184.4 KB


Comprobar que no faltan fechas

In [30]:
# 1. Cargar el archivo CSV
df = pd.read_csv("generacion.csv")

# Sustituya esto por el nombre real de su columna de fechas
columna_fecha = "fecha" 

# 2. Convertir la columna a formato datetime
# Si su función formatear_df guardó las fechas como 'DD/MM/YYYY', use dayfirst=True
df[columna_fecha] = pd.to_datetime(df[columna_fecha], format='%d/%m/%Y', errors='coerce')

# 3. Extraer las fechas únicas encontradas en el CSV (ignorando la hora si la tuviera)
fechas_extraidas = pd.to_datetime(df[columna_fecha].dt.date.dropna().unique())

# 4. Generar el rango de fechas esperado
fechas_esperadas = pd.date_range(start="2021-01-01", end="2026-05-21")

# 5. Calcular la diferencia matemática (Fechas Esperadas - Fechas Extraídas)
fechas_faltantes = fechas_esperadas.difference(fechas_extraidas)

# 6. Mostrar los resultados
if len(fechas_faltantes) == 0:
    print("Validación completada: No falta ninguna fecha en el rango especificado.")
else:
    print(f"Alerta: Faltan {len(fechas_faltantes)} fechas en el CSV.")
    print("Lista de fechas faltantes:")
    for fecha in fechas_faltantes:
        print(fecha.date())

Validación completada: No falta ninguna fecha en el rango especificado.
